<a href="https://colab.research.google.com/github/pradervonsky/vbig-lab/blob/main/evaluation/evaluation_L2-L3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Small VLMs Evaluation Pipeline

## Initial steps

In [1]:
!pip install --q supabase rouge-score bert-score anthropic

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 700.6/700.6 kB 23.6 MB/s eta 0:00:00


In [2]:
import re
import time
import json
import numpy as np
import pandas as pd
from google.colab import userdata
from supabase import create_client

In [3]:
SUPABASE_URL = userdata.get("SUPABASE_URL")
SUPABASE_KEY = userdata.get("SUPABASE_KEY")
supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

### Load tables

In [4]:
# human_insights: primary annotator ground truth (insight_part_1)
human_resp = supabase.table("human_insights").select("*").execute()
df_human_raw = pd.DataFrame(human_resp.data)

# vlm_outputs: all 8 models × all dashboards
vlm_resp = supabase.table("vlm_outputs").select("*").execute()
df_vlm_raw = pd.DataFrame(vlm_resp.data)

# metadata: dashboard_name lookup
meta_resp = supabase.table("metadata").select("id, dashboard_name").execute()
df_meta = pd.DataFrame(meta_resp.data).rename(columns={"id": "metadata_id"})

print(f"human_insights rows : {len(df_human_raw)}")
print(f"vlm_outputs rows    : {len(df_vlm_raw)}")
print(f"metadata rows       : {len(df_meta)}")

human_insights rows : 271
vlm_outputs rows    : 320
metadata rows       : 270


In [5]:
# Keep only rows that are in the final eval corpus:
# expected_dataset=True AND rejection_reason IS NULL
df_human = df_human_raw[
    (df_human_raw["expected_dataset"] == True) &
    (df_human_raw["rejection_reason"].isna())
].copy()

# Attach dashboard_name
df_human = df_human.merge(df_meta, on="metadata_id", how="left")

print(f"Eval dashboards (human_insights): {len(df_human)}")
assert len(df_human) == 40, f"Expected 40, got {len(df_human)}"

Eval dashboards (human_insights): 40


### Parsing helpers

In [6]:
NA_PATTERN = re.compile(r"^(not applicable|n/a)$", re.IGNORECASE)

def normalize_value(val):
    if val is None:
        return None
    cleaned = str(val).strip().rstrip(".").strip()
    if cleaned == "" or NA_PATTERN.match(cleaned):
        return None
    return cleaned

def parse_charts(text):
    """
    Parse a free-text blob into a list of chart dicts with keys:
    chart_id (int), title (str), L2, L3, L4 (str or None).

    Handles variants:
      - 'Chart N: Title'  (standard)
      - 'Chat N: Title'   (typo)
      - 'Chart N : Title' (extra space before colon)
    """
    charts = []
    if not text or (isinstance(text, float) and np.isnan(text)):
        return charts

    # Split on any Chart/Chat header; keep delimiter via lookahead
    blocks = re.split(r"(?=(?:Chart|Chat)\s+\d+\s*[:.])", str(text).strip())

    for block in blocks:
        block = block.strip()
        if not block:
            continue
        lines = block.splitlines()
        header = lines[0].strip()
        match = re.match(r"(?:Chart|Chat)\s+(\d+)\s*[:.]\s*(.*)", header)
        if not match:
            continue
        chart_id = int(match.group(1))
        title    = match.group(2).strip()

        L2 = L3 = L4 = None
        for line in lines[1:]:
            line = line.strip()
            if line.startswith("L2:"):
                L2 = normalize_value(line[3:].strip())
            elif line.startswith("L3:"):
                L3 = normalize_value(line[3:].strip())
            elif line.startswith("L4:"):
                L4 = normalize_value(line[3:].strip())

        charts.append({"chart_id": chart_id, "title": title, "L2": L2, "L3": L3, "L4": L4})
    return charts

### Build the ground truth df


In [7]:
# One row per (dashboard × chart × level)
# Reference text = insight_part_1 (primary annotator)

gt_records = []
for _, row in df_human.iterrows():
    metadata_id    = row["metadata_id"]
    dashboard_name = row["dashboard_name"]
    human_id       = row["id"]

    parsed = parse_charts(row["insight_part_1"])

    for chart in parsed:
        for level in ["L2", "L3", "L4"]:
            gt_records.append({
                "human_id"      : human_id,
                "metadata_id"   : metadata_id,
                "dashboard_name": dashboard_name,
                "chart_id"      : chart["chart_id"],
                "chart_title"   : chart["title"],
                "level"         : level,
                "reference"     : chart[level],   # None if NA/empty
            })

df_gt = pd.DataFrame(gt_records)
print(f"Ground truth units : {len(df_gt)}")
print(df_gt.head(9).to_string())

Ground truth units : 612
                               human_id                           metadata_id               dashboard_name  chart_id          chart_title level                                                                                                                                                                                               reference
0  c1c00ab2-139a-4712-9fea-8e87f6d7e70c  ba445ab0-8ff3-45ad-8cb5-ec7275baab13  Superstore - Bento Box KPIs         1  Scoreboard Overview    L2            Corporate sales are $248,244.1, consumer sales are $334,911.9, and home office sales are $162,411.5, cumulating to $745,568 total sales with profit of $95,926 and the profit ratio of 12.9%
1  c1c00ab2-139a-4712-9fea-8e87f6d7e70c  ba445ab0-8ff3-45ad-8cb5-ec7275baab13  Superstore - Bento Box KPIs         1  Scoreboard Overview    L3                                                                                                                                                

### Build the VLM outputs df

In [8]:
# Filter to eval dashboards only (inner join on metadata_id present in df_human)
eval_metadata_ids = set(df_human["metadata_id"].unique())

df_vlm = df_vlm_raw[df_vlm_raw["metadata_id"].isin(eval_metadata_ids)].copy()
df_vlm = df_vlm.merge(df_meta, on="metadata_id", how="left")

vlm_records = []
for _, row in df_vlm.iterrows():
    metadata_id    = row["metadata_id"]
    dashboard_name = row["dashboard_name"]
    model_name     = row["model_name"]
    model_hf_id    = row["model_hf_id"]
    inference_ms   = row.get("inference_ms", None)

    parsed = parse_charts(row["raw_output"])

    for chart in parsed:
        for level in ["L2", "L3", "L4"]:
            vlm_records.append({
                "metadata_id"   : metadata_id,
                "dashboard_name": dashboard_name,
                "model_name"    : model_name,
                "model_hf_id"   : model_hf_id,
                "chart_id"      : chart["chart_id"],
                "chart_title"   : chart["title"],
                "level"         : level,
                "generated"     : chart[level],   # None if NA/empty or unparseable
                "inference_ms"  : inference_ms,
            })

df_vlm_long = pd.DataFrame(vlm_records)
print(f"VLM output units : {len(df_vlm_long)}")
print(df_vlm_long[["model_name","dashboard_name","chart_id","level","generated"]].head(12).to_string())

VLM output units : 6798
              model_name             dashboard_name  chart_id level                                                                                                        generated
0   Qwen3-VL-2B-Instruct  Superstore Sales Overview         1    L2                                                      Sales $733.2K, Profit $93.4K, Orders 1,687, Quantity 12,476
1   Qwen3-VL-2B-Instruct  Superstore Sales Overview         1    L3                                    All KPIs show positive growth, with sales and profit increasing significantly
2   Qwen3-VL-2B-Instruct  Superstore Sales Overview         1    L4  The overall performance is strong, with sales and profit increasing substantially compared to the previous year
3   Qwen3-VL-2B-Instruct  Superstore Sales Overview         2    L2                                                 Total Sales $733.2K, Target Sales $1.0M, Percent of Target 73.3%
4   Qwen3-VL-2B-Instruct  Superstore Sales Overview         2    L3    

In [9]:
# Deduplicate on conflict key
df_vlm_long = df_vlm_long.drop_duplicates(
    subset=["model_name", "metadata_id", "chart_id", "level"],
    keep="last"
).reset_index(drop=True)

print(f"VLM output units after dedup : {len(df_vlm_long)}")

VLM output units after dedup : 6750


### Merge df

In [10]:
# Join on metadata_id + chart_id + level
df_eval = df_vlm_long.merge(
    df_gt[["metadata_id", "chart_id", "level", "reference"]],
    on=["metadata_id", "chart_id", "level"],
    how="left"
)

# Parseability flag: True if generated text is non-None after parse_charts
df_eval["generation_success"] = df_eval["generated"].notna()

print(f"Eval pairs total : {len(df_eval)}")

Eval pairs total : 6750


In [11]:
stat_rows = []
for model in sorted(df_eval["model_name"].unique()):
    m = df_eval[df_eval["model_name"] == model]
    stat_rows.append({
        "model_name"      : model,
        "total_dashboards": m["metadata_id"].nunique(),
        "total_charts"    : m.groupby("metadata_id")["chart_id"].nunique().sum(),
        "total_levels"    : len(m),
    })

# Grand total row
stat_rows.append({
    "model_name"      : "GRAND TOTAL",
    "total_dashboards": df_eval["metadata_id"].nunique(),
    "total_charts"    : df_eval.groupby(["model_name","metadata_id"])["chart_id"].nunique().sum(),
    "total_levels"    : len(df_eval),
})

df_stats = pd.DataFrame(stat_rows)
print("=== Per-model statistics ===")
print(df_stats.to_string(index=False))

=== Per-model statistics ===
           model_name  total_dashboards  total_charts  total_levels
         InternVL2-1B                40           622          1866
         InternVL2-2B                40           240           720
      InternVL3-1B-hf                40           353          1059
 Qwen3-VL-2B-Instruct                40           201           603
         Qwen3.5-0.8B                40           353          1059
           Qwen3.5-2B                40           206           618
SmolVLM-256M-Instruct                40           151           453
           moondream2                39           124           372
          GRAND TOTAL                40          2250          6750


## Deployment summary

### VLMs generation summary (4.1)

In [12]:
# Per model: dashboards attempted, parseability rate, avg inference time
# Also flag degenerate patterns in raw_output

DEGENERATE_PATTERNS = {
    "prompt-echo": re.compile(r"You are a senior BI analyst|generate insights for a business intelligence", re.IGNORECASE),
    "repetition" : re.compile(r"(.{20,})\1{2,}", re.DOTALL),
    "empty"      : re.compile(r"^\s*$"),
}

def classify_degenerate(raw_text):
    if raw_text is None or (isinstance(raw_text, float) and np.isnan(raw_text)):
        return "empty"
    for label, pattern in DEGENERATE_PATTERNS.items():
        if pattern.search(str(raw_text)):
            return label
    return None

df_vlm_raw_eval = df_vlm_raw[df_vlm_raw["metadata_id"].isin(eval_metadata_ids)].copy()
df_vlm_raw_eval = df_vlm_raw_eval.merge(df_meta, on="metadata_id", how="left")
df_vlm_raw_eval["degenerate_pattern"] = df_vlm_raw_eval["raw_output"].apply(classify_degenerate)
df_vlm_raw_eval["parseable"] = df_vlm_raw_eval["raw_output"].apply(
    lambda x: len(parse_charts(x)) > 0
)

deployment_summary = (
    df_vlm_raw_eval
    .groupby("model_name")
    .agg(
        dashboards_completed = ("metadata_id", "nunique"),
        parseability_rate    = ("parseable", "mean"),
        avg_inference_ms     = ("inference_ms", "mean"),
        degenerate_any       = ("degenerate_pattern", lambda x: x.notna().sum()),
    )
    .reset_index()
)

# Paired count statistics
# Built from df_eval_all: the merged DataFrame before SmolVLM exclusion
# so all 8 models are represented here for the deployment overview.
# df_eval at this point still contains all models (exclusion happens later).

total_gt_keys = len(df_gt[["metadata_id","chart_id","level"]].drop_duplicates())

paired_counts = (
    df_eval[df_eval["generated"].notna() & df_eval["reference"].notna()]
    .groupby("model_name").size().reset_index(name="paired_count")
)
missing_vlm = (
    df_eval[df_eval["generated"].isna() & df_eval["reference"].notna()]
    .groupby("model_name").size().reset_index(name="missing_vlm")
)
missing_gt = (
    df_eval[df_eval["generated"].notna() & df_eval["reference"].isna()]
    .groupby("model_name").size().reset_index(name="missing_gt")
)

deployment_summary = (
    deployment_summary
    .merge(paired_counts, on="model_name", how="left")
    .merge(missing_vlm,   on="model_name", how="left")
    .merge(missing_gt,    on="model_name", how="left")
    .fillna(0)
)

for col in ["paired_count", "missing_vlm", "missing_gt"]:
    deployment_summary[col] = deployment_summary[col].astype(int)

deployment_summary["paired_pct"] = (
    deployment_summary["paired_count"] / total_gt_keys * 100
).round(1).astype(str) + "%"

# Format display columns
deployment_summary["parseability_rate"] = deployment_summary["parseability_rate"].map("{:.1%}".format)
deployment_summary["avg_inference_ms"]  = deployment_summary["avg_inference_ms"].map("{:,.0f}".format)

print(f"Total GT keys (chart × level, all 3 levels): {total_gt_keys}")
print("\n=== Deployment Summary ===")
print(deployment_summary[[
    "model_name", "dashboards_completed", "parseability_rate",
    "avg_inference_ms", "degenerate_any",
    "paired_count", "paired_pct", "missing_vlm", "missing_gt"
]].to_string(index=False))

Total GT keys (chart × level, all 3 levels): 612

=== Deployment Summary ===
           model_name  dashboards_completed parseability_rate avg_inference_ms  degenerate_any  paired_count paired_pct  missing_vlm  missing_gt
         InternVL2-1B                    40            100.0%           39,556               0           342      55.9%          143         756
         InternVL2-2B                    40            100.0%           26,436               1           461      75.3%            0         255
      InternVL3-1B-hf                    40            100.0%           40,605               8           490      80.1%            4         521
 Qwen3-VL-2B-Instruct                    40            100.0%           78,069               1           520      85.0%            0          81
         Qwen3.5-0.8B                    40            100.0%           74,595               0           559      91.3%            0         486
           Qwen3.5-2B                    40          

In [13]:
# Sanity check: how many lines contain L2/L3/L4 but are not being parsed?
unparsed = []
for _, row in df_vlm_raw_eval.iterrows():
    if not row["raw_output"]:
        continue
    for line in str(row["raw_output"]).splitlines():
        line = line.strip()
        if re.match(r"L[234]\s+:", line):   # catches "L2 : " variant
            unparsed.append({
                "model_name": row["model_name"],
                "dashboard_name": row["dashboard_name"],
                "line": line[:100]
            })

print(f"Lines with 'L2/3/4 :' spacing variant: {len(unparsed)}")
if unparsed:
    pd.DataFrame(unparsed).head(20)

Lines with 'L2/3/4 :' spacing variant: 0


In [14]:
# Quick check: what strings do VLMs actually write for L3 NA cases?

l3_nulls = df_vlm_long[
    (df_vlm_long["level"] == "L3") &
    (df_vlm_long["generated"].isna())
]
print(f"L3 rows currently parsed as None: {len(l3_nulls)}")

# Also check raw parsed values before normalization to see the actual strings
# Re-parse one dashboard to inspect
sample_raw = df_vlm_raw_eval[df_vlm_raw_eval["model_name"] == "Qwen3.5-2B"].iloc[0]["raw_output"]
print("\nSample raw L3 values from Qwen3.5-2B:")
for chart in parse_charts(sample_raw):
    print(f"  Chart {chart['chart_id']} L3: {repr(chart['L3'])}")

L3 rows currently parsed as None: 291

Sample raw L3 values from Qwen3.5-2B:
  Chart 1 L3: 'The sales trend line appears to be volatile, showing significant spikes and drops across the year'
  Chart 2 L3: 'The map shows a concentrated cluster of sales in the West and East Coast, with minimal activity in the Midwest'
  Chart 3 L3: 'The bar chart shows a wide spread between the current year and previous year values for most categories'
  Chart 4 L3: 'The pie chart indicates that the Consumer segment holds the largest portion of the total sales volume'


### Identify which chart has no counterpart between GT and VLMs (4.4)

In [15]:
# Missing from VLM: GT key has no VLM counterpart (per model, aggregated)
gt_keys_full = df_gt[["metadata_id","dashboard_name","chart_id","level"]].drop_duplicates()
vlm_keys_full = df_vlm_long[["metadata_id","dashboard_name","model_name","chart_id","level"]].drop_duplicates()

all_models = df_vlm_long["model_name"].unique()
expected_rows = []
for model in all_models:
    tmp = gt_keys_full.copy()
    tmp["model_name"] = model
    expected_rows.append(tmp)
expected_grid = pd.concat(expected_rows, ignore_index=True)

mismatch = expected_grid.merge(
    vlm_keys_full.assign(in_vlm=True),
    on=["metadata_id","dashboard_name","model_name","chart_id","level"],
    how="left"
)
mismatch["in_vlm"] = mismatch["in_vlm"].fillna(False)
missing_from_vlm = mismatch[~mismatch["in_vlm"]]

table1 = (
    missing_from_vlm
    .groupby("model_name")
    .size()
    .reset_index(name="missing_gt_keys")
    .sort_values("missing_gt_keys", ascending=False)
)
total_gt_keys = len(gt_keys_full)
table1["pct_of_expected"] = (table1["missing_gt_keys"] / (total_gt_keys * 1) * 100).round(1)

print("=== GT keys with no VLM counterpart (per model) ===")
print(f"Total GT keys (chart × level): {total_gt_keys}")
print(table1.to_string(index=False))


=== GT keys with no VLM counterpart (per model) ===
Total GT keys (chart × level): 612
           model_name  missing_gt_keys  pct_of_expected
SmolVLM-256M-Instruct              324             52.9
           moondream2              321             52.5
         InternVL2-2B              126             20.6
         InternVL2-1B               99             16.2
      InternVL3-1B-hf               87             14.2
 Qwen3-VL-2B-Instruct               63             10.3
           Qwen3.5-2B               48              7.8
         Qwen3.5-0.8B               15              2.5


/tmp/ipykernel_16572/1760942875.py:18: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  mismatch["in_vlm"] = mismatch["in_vlm"].fillna(False)


In [16]:
# VLM outputs with no GT counterpart (per model, aggregated)
extra_in_vlm = vlm_keys_full.merge(
    gt_keys_full.assign(in_gt=True),
    on=["metadata_id","dashboard_name","chart_id","level"],
    how="left"
)
extra_in_vlm["in_gt"] = extra_in_vlm["in_gt"].fillna(False)
hallucinated = extra_in_vlm[~extra_in_vlm["in_gt"]]

table2 = (
    hallucinated
    .groupby("model_name")
    .size()
    .reset_index(name="hallucinated_keys")
    .sort_values("hallucinated_keys", ascending=False)
)
total_vlm_keys = len(vlm_keys_full)
table2["pct_of_vlm_output"] = (table2["hallucinated_keys"] / total_vlm_keys * 100).round(1)

print("\n=== VLM outputs with no GT counterpart (per model) ===")
print(f"Total VLM output keys (chart × level): {total_vlm_keys}")
print(table2.to_string(index=False))


=== VLM outputs with no GT counterpart (per model) ===
Total VLM output keys (chart × level): 6750
           model_name  hallucinated_keys  pct_of_vlm_output
         InternVL2-1B               1353               20.0
      InternVL3-1B-hf                534                7.9
         Qwen3.5-0.8B                462                6.8
         InternVL2-2B                234                3.5
SmolVLM-256M-Instruct                165                2.4
           moondream2                 81                1.2
 Qwen3-VL-2B-Instruct                 54                0.8
           Qwen3.5-2B                 54                0.8


/tmp/ipykernel_16572/4015012248.py:7: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  extra_in_vlm["in_gt"] = extra_in_vlm["in_gt"].fillna(False)


### Scoreboard Overview missingness (4.4)

In [17]:
# VLM Chart 1 ≠ "Scoreboard Overview" where GT Chart 1 IS "Scoreboard Overview"

# GT: find dashboards where chart_id==1 title is "Scoreboard Overview"
gt_chart1 = df_gt[df_gt["chart_id"] == 1][["metadata_id","dashboard_name","chart_title"]].drop_duplicates()
gt_scoreboard_ids = set(
    gt_chart1[gt_chart1["chart_title"].str.strip().str.lower() == "scoreboard overview"]["metadata_id"]
)

print(f"\n=== Scoreboard Overview alignment ===")
print(f"Dashboards where GT Chart 1 = 'Scoreboard Overview': {len(gt_scoreboard_ids)} / {df_gt['metadata_id'].nunique()}")

# VLM: for those dashboards, what is VLM's chart 1 title?
vlm_chart1 = (
    df_vlm_long[
        (df_vlm_long["chart_id"] == 1) &
        (df_vlm_long["metadata_id"].isin(gt_scoreboard_ids))
    ]
    [["metadata_id","dashboard_name","model_name","chart_title"]]
    .drop_duplicates()
)

vlm_chart1["is_scoreboard"] = (
    vlm_chart1["chart_title"].str.strip().str.lower() == "scoreboard overview"
)

table4 = (
    vlm_chart1
    .groupby("model_name")
    .agg(
        total_applicable   = ("metadata_id", "nunique"),
        scoreboard_correct = ("is_scoreboard", "sum"),
    )
    .reset_index()
)
table4["missing_scoreboard"] = table4["total_applicable"] - table4["scoreboard_correct"]
table4["pct_missing"] = (table4["missing_scoreboard"] / table4["total_applicable"] * 100).round(1)

print(table4.to_string(index=False))


=== Scoreboard Overview alignment ===
Dashboards where GT Chart 1 = 'Scoreboard Overview': 35 / 40
           model_name  total_applicable  scoreboard_correct  missing_scoreboard  pct_missing
         InternVL2-1B                35                   0                  35        100.0
         InternVL2-2B                35                   0                  35        100.0
      InternVL3-1B-hf                35                   0                  35        100.0
 Qwen3-VL-2B-Instruct                35                  34                   1          2.9
         Qwen3.5-0.8B                35                   0                  35        100.0
           Qwen3.5-2B                35                  32                   3          8.6
SmolVLM-256M-Instruct                35                   0                  35        100.0
           moondream2                34                   0                  34        100.0


### Degenerate VLMs output analysis


*   Degenerate pattern distribution table (4.1)
*   Degenerate sample outputs (4.5)



In [18]:
# Degenerate pattern distribution
degen_dist = (
    df_vlm_raw_eval
    .groupby(["model_name", "degenerate_pattern"])
    .size()
    .reset_index(name="count")
)

all_model_degen = (
    df_vlm_raw_eval
    .groupby("model_name")["degenerate_pattern"]
    .apply(lambda x: x.notna().sum())
    .reset_index(name="total_degenerate")
)

degen_pivot = (
    degen_dist[degen_dist["degenerate_pattern"].notna()]
    .pivot(index="model_name", columns="degenerate_pattern", values="count")
    .fillna(0).astype(int)
    .reset_index()
)

degen_pivot = all_model_degen.merge(degen_pivot, on="model_name", how="left")
degen_pivot["total_degenerate"] = degen_pivot["total_degenerate"].astype(int)

# Fill any missing pattern columns with 0
for col in ["repetition", "prompt-echo", "empty"]:
    if col not in degen_pivot.columns:
        degen_pivot[col] = 0
    else:
        degen_pivot[col] = degen_pivot[col].fillna(0).astype(int)

grand_total = len(df_vlm_raw_eval["metadata_id"].unique()) * 0 + len(df_vlm_raw_eval)  # total rows per model summed
degen_pivot["total_correct"] = grand_total // len(degen_pivot) - degen_pivot["total_degenerate"]

# Percentages against grand total per model (40 dashboards)
total_per_model = 40
degen_pivot["pct_degenerate"] = (degen_pivot["total_degenerate"] / total_per_model * 100).round(1).astype(str) + "%"
degen_pivot["pct_correct"]    = (degen_pivot["total_correct"]    / total_per_model * 100).round(1).astype(str) + "%"

print("=== Degenerate pattern distribution (per model) ===")
print(f"Grand total per model: {total_per_model} dashboards\n")
print(degen_pivot[["model_name", "repetition", "prompt-echo", "empty",
                    "total_degenerate", "pct_degenerate",
                    "total_correct",    "pct_correct"]].to_string(index=False))

=== Degenerate pattern distribution (per model) ===
Grand total per model: 40 dashboards

           model_name  repetition  prompt-echo  empty  total_degenerate pct_degenerate  total_correct pct_correct
         InternVL2-1B           0            0      0                 0           0.0%             40      100.0%
         InternVL2-2B           1            0      0                 1           2.5%             39       97.5%
      InternVL3-1B-hf           8            0      0                 8          20.0%             32       80.0%
 Qwen3-VL-2B-Instruct           1            0      0                 1           2.5%             39       97.5%
         Qwen3.5-0.8B           0            0      0                 0           0.0%             40      100.0%
           Qwen3.5-2B           0            0      0                 0           0.0%             40      100.0%
SmolVLM-256M-Instruct           0           40      0                40         100.0%              0        0.0

In [19]:
# Inspect degenerate outputs, 1 sample per model

print("=== Degenerate outputs (1 sample per model) ===\n")
degen_rows = df_vlm_raw_eval[df_vlm_raw_eval["degenerate_pattern"].notna()].copy()

if degen_rows.empty:
    print("None found.")
else:
    for model, group in degen_rows.groupby("model_name"):
        r = group.iloc[0]
        print(f"Model    : {r['model_name']}")
        print(f"Dashboard: {r['dashboard_name']}")
        print(f"Pattern  : {r['degenerate_pattern']}")
        print(f"Raw output (first 400 chars):\n{str(r['raw_output'])[:400]}")
        print("-" * 70)

=== Degenerate outputs (1 sample per model) ===

Model    : InternVL2-2B
Dashboard: SUPERSTORE DASHBOARD
Pattern  : repetition
Raw output (first 400 chars):
Chart 1: Sales Overview
L2: Sales €733.2K vs 20.4% vs PY
L3: Sales by Segment €331.9K vs €241.8K vs €159.5K vs €246.1K vs €215.4K vs €93.4K vs €14.2K vs PY
L4: Sales by Category Consumer, Corporate, Home Office, Technology, Office Supplies, Furniture vs €241.8K vs €159.5K vs €246.1K vs €215.4K vs €93.4K vs €14.2K vs PY

Chart 2: Profit Overview
L2: Profit €93.4K vs 14.2% vs PY
L3: Profit by Segmen
----------------------------------------------------------------------
Model    : InternVL3-1B-hf
Dashboard: Mobile-Optimized Superstore Sales Dashboard
Pattern  : repetition
Raw output (first 400 chars):
Chart 1: TOTAL SALES
L2: The total sales for the year 2021 are $733,215, with a 20.4% YoY growth from the previous year.
L3: The total sales have increased from $733,215 in January to $93,439 in December, showing a 14.2% YoY growth.
L4: 

In [20]:
# moondream2 97.5% parseability: find the 1 failing dashboard
moon_rows = df_vlm_raw_eval[df_vlm_raw_eval["model_name"] == "moondream2"].copy()
moon_rows["parseable"] = moon_rows["raw_output"].apply(lambda x: len(parse_charts(x)) > 0)

unparseable = moon_rows[~moon_rows["parseable"]]

print(f"moondream2 total rows   : {len(moon_rows)}")
print(f"moondream2 unparseable  : {len(unparseable)}")
print()
for _, r in unparseable.iterrows():
    print(f"Dashboard  : {r['dashboard_name']}")
    print(f"metadata_id: {r['metadata_id']}")
    print(f"Raw output :\n{r['raw_output']}")
    print("-" * 70)

moondream2 total rows   : 40
moondream2 unparseable  : 1

Dashboard  : SUPERSTORE DASHBOARD
metadata_id: 8d8d0715-572a-44c1-850d-287d7069ff71
Raw output :
Chart count: N
L2: Volatile
L3: appears to
L4: appears to
----------------------------------------------------------------------


## Evaluation

### ROUGE (L2 & L3)

In [21]:
from rouge_score import rouge_scorer as rs
import math

rouge = rs.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=False)

def is_valid_text(val):
    if val is None:
        return False
    if isinstance(val, float) and math.isnan(val):
        return False
    return str(val).strip() != ""

def compute_rouge(generated, reference):
    if not is_valid_text(generated) or not is_valid_text(reference):
        return {"rouge1": np.nan, "rouge2": np.nan, "rougeL": np.nan}
    scores = rouge.score(str(reference), str(generated))
    return {
        "rouge1": scores["rouge1"].fmeasure,
        "rouge2": scores["rouge2"].fmeasure,
        "rougeL": scores["rougeL"].fmeasure,
    }

# Apply only to L2 and L3
mask_rouge = df_eval["level"].isin(["L2", "L3"])

rouge_scores = df_eval[mask_rouge].apply(
    lambda r: compute_rouge(r["generated"], r["reference"]), axis=1
)
rouge_df = pd.DataFrame(rouge_scores.tolist(), index=df_eval[mask_rouge].index)

df_eval.loc[mask_rouge, "rouge1"] = rouge_df["rouge1"]
df_eval.loc[mask_rouge, "rouge2"] = rouge_df["rouge2"]
df_eval.loc[mask_rouge, "rougeL"] = rouge_df["rougeL"]

for col in ["rouge1", "rouge2", "rougeL"]:
    df_eval.loc[~mask_rouge, col] = np.nan

print(df_eval[mask_rouge][["model_name","level","rouge1","rouge2","rougeL"]].describe())

            rouge1       rouge2       rougeL
count  2241.000000  2241.000000  2241.000000
mean      0.198985     0.053472     0.166691
std       0.156555     0.088908     0.133299
min       0.000000     0.000000     0.000000
25%       0.080000     0.000000     0.071429
50%       0.186047     0.000000     0.156863
75%       0.285714     0.081633     0.235294
max       0.827586     0.628571     0.827586


### BERTScore (L2 & L3, rescaled, roberta-large)

In [22]:
from bert_score import score as bert_score_fn

def compute_bertscore_batch(df_subset):
    results = pd.Series(np.nan, index=df_subset.index)
    valid   = df_subset[df_subset["generated"].notna() & df_subset["reference"].notna()]

    if valid.empty:
        return results

    cands = valid["generated"].tolist()
    refs  = valid["reference"].tolist()

    _, _, F1 = bert_score_fn(
        cands, refs,
        model_type="roberta-large",
        lang="en",
        rescale_with_baseline=True,
        verbose=False,
    )
    results.loc[valid.index] = F1.numpy()
    return results

# Run BERTScore per level to avoid memory spikes
for level in ["L2", "L3"]:
    subset = df_eval[df_eval["level"] == level].copy()
    scores = compute_bertscore_batch(subset)
    df_eval.loc[scores.index, "bertscore_f1"] = scores

df_eval.loc[df_eval["level"] == "L4", "bertscore_f1"] = np.nan

print(df_eval[df_eval["level"].isin(["L2","L3"])][["model_name","level","bertscore_f1"]].describe())

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


       bertscore_f1
count   2241.000000
mean       0.183120
std        0.201745
min       -0.403514
25%        0.061173
50%        0.204191
75%        0.318673
max        0.800141


## Evaluation results

### Across semantic levels

In [23]:
# Aggregate across all models × dashboards, per level
# ROUGE-1, ROUGE-2, ROUGE-L and BERTScore for L2/L3; G-Eval for L3/L4
# ROUGE-L is the primary metric (marked with * in output)

def level_summary(df, metric_col, levels):
    sub = df[df["level"].isin(levels) & df[metric_col].notna()]
    return sub.groupby("level")[metric_col].agg(mean="mean", sd="std").round(4)

rouge1_summary = level_summary(df_eval, "rouge1",       ["L2","L3"])
rouge2_summary = level_summary(df_eval, "rouge2",       ["L2","L3"])
rougeL_summary = level_summary(df_eval, "rougeL",       ["L2","L3"])
bert_summary   = level_summary(df_eval, "bertscore_f1", ["L2","L3"])

table_43 = (
    rouge1_summary.rename(columns={"mean":"ROUGE-1 mean","sd":"ROUGE-1 SD"})
    .join(rouge2_summary.rename(columns={"mean":"ROUGE-2 mean","sd":"ROUGE-2 SD"}), how="outer")
    .join(rougeL_summary.rename(columns={"mean":"ROUGE-L* mean","sd":"ROUGE-L* SD"}), how="outer")
    .join(bert_summary.rename(columns={"mean":"BERTScore mean","sd":"BERTScore SD"}), how="outer")
    .reindex(["L2","L3"])
)

print("\n=== Performance Across Semantic Levels ===")
print("(* = primary metric)")
print(table_43.to_string())


=== Performance Across Semantic Levels ===
(* = primary metric)
       ROUGE-1 mean  ROUGE-1 SD  ROUGE-2 mean  ROUGE-2 SD  ROUGE-L* mean  ROUGE-L* SD  BERTScore mean  BERTScore SD
level                                                                                                              
L2           0.2121      0.1798        0.0728      0.1049         0.1818       0.1562          0.1878        0.2220
L3           0.1829      0.1205        0.0299      0.0559         0.1483       0.0952          0.1774        0.1739


### Main model x metric table

In [24]:
def fmt(mean, sd):
    if pd.isna(mean):
        return "-"
    return f"{mean:.3f} ± {sd:.3f}"

rows = []
for model in df_eval["model_name"].unique():
    m = df_eval[df_eval["model_name"] == model]

    def ms(level, metric):
        sub = m[(m["level"]==level) & m[metric].notna()][metric]
        return sub.mean(), sub.std()

    l2_r1_m, l2_r1_s = ms("L2", "rouge1")
    l2_r2_m, l2_r2_s = ms("L2", "rouge2")
    l2_rl_m, l2_rl_s = ms("L2", "rougeL")
    l2_b_m,  l2_b_s  = ms("L2", "bertscore_f1")
    l3_r1_m, l3_r1_s = ms("L3", "rouge1")
    l3_r2_m, l3_r2_s = ms("L3", "rouge2")
    l3_rl_m, l3_rl_s = ms("L3", "rougeL")
    l3_b_m,  l3_b_s  = ms("L3", "bertscore_f1")

    rows.append({
        "Model"        : model,
        "L2 ROUGE-1"   : fmt(l2_r1_m, l2_r1_s),
        "L2 ROUGE-2"   : fmt(l2_r2_m, l2_r2_s),
        "L2 ROUGE-L*"  : fmt(l2_rl_m, l2_rl_s),
        "L2 BERTScore" : fmt(l2_b_m,  l2_b_s),
        "L3 ROUGE-1"   : fmt(l3_r1_m, l3_r1_s),
        "L3 ROUGE-2"   : fmt(l3_r2_m, l3_r2_s),
        "L3 ROUGE-L*"  : fmt(l3_rl_m, l3_rl_s),
        "L3 BERTScore" : fmt(l3_b_m,  l3_b_s),
    })

table_44 = pd.DataFrame(rows).set_index("Model")
print("\n=== Model × Metric Performance (mean ± SD) ===")
print("(* = primary metric)")
print(table_44.to_string())


=== Model × Metric Performance (mean ± SD) ===
(* = primary metric)
                          L2 ROUGE-1     L2 ROUGE-2    L2 ROUGE-L*    L2 BERTScore     L3 ROUGE-1     L3 ROUGE-2    L3 ROUGE-L*    L3 BERTScore
Model                                                                                                                                          
Qwen3-VL-2B-Instruct   0.325 ± 0.191  0.135 ± 0.131  0.278 ± 0.173   0.315 ± 0.213  0.249 ± 0.118  0.062 ± 0.091  0.193 ± 0.099   0.286 ± 0.139
moondream2             0.147 ± 0.130  0.028 ± 0.069  0.126 ± 0.112   0.046 ± 0.196  0.118 ± 0.088  0.006 ± 0.021  0.102 ± 0.069   0.109 ± 0.130
Qwen3.5-0.8B           0.139 ± 0.161  0.048 ± 0.091  0.125 ± 0.147   0.159 ± 0.190  0.223 ± 0.089  0.040 ± 0.057  0.187 ± 0.075   0.232 ± 0.096
Qwen3.5-2B             0.366 ± 0.172  0.143 ± 0.127  0.300 ± 0.154   0.347 ± 0.155  0.256 ± 0.093  0.039 ± 0.049  0.194 ± 0.062   0.274 ± 0.097
InternVL2-1B           0.131 ± 0.131  0.038 ± 0.073  0.127 ± 0.126 

### Within-family analysis

In [25]:
# Compare within-family pairs:
# Qwen3.5: 0.8B vs 2B
# InternVL2: 1B vs 2B

FAMILY_PAIRS = {
    "Qwen3.5"  : ("Qwen3.5-0.8B", "Qwen3.5-2B"),
    "InternVL2": ("InternVL2-1B",  "InternVL2-2B"),
}

METRICS = {
    "L2": ["rouge1", "rouge2", "rougeL", "bertscore_f1"],
    "L3": ["rouge1", "rouge2", "rougeL", "bertscore_f1"],
}

scaling_rows = []
for family, (small, large) in FAMILY_PAIRS.items():
    for level, metrics in METRICS.items():
        for metric in metrics:
            sub = df_eval[df_eval["level"] == level]
            small_mean = sub[sub["model_name"]==small][metric].mean()
            large_mean = sub[sub["model_name"]==large][metric].mean()
            delta = round(large_mean - small_mean, 4) if not (np.isnan(small_mean) or np.isnan(large_mean)) else np.nan
            scaling_rows.append({
                "Family" : family,
                "Level"  : level,
                "Metric" : metric,
                "Small"  : round(small_mean, 4),
                "Large"  : round(large_mean, 4),
                "Delta"  : delta,
            })

df_scaling = pd.DataFrame(scaling_rows)
print("\n=== Within-Family Parameter Scaling ===")
print(df_scaling.to_string(index=False))


=== Within-Family Parameter Scaling ===
   Family Level       Metric  Small  Large   Delta
  Qwen3.5    L2       rouge1 0.1393 0.3659  0.2266
  Qwen3.5    L2       rouge2 0.0478 0.1434  0.0956
  Qwen3.5    L2       rougeL 0.1247 0.2996  0.1749
  Qwen3.5    L2 bertscore_f1 0.1588 0.3471  0.1884
  Qwen3.5    L3       rouge1 0.2231 0.2555  0.0324
  Qwen3.5    L3       rouge2 0.0396 0.0389 -0.0007
  Qwen3.5    L3       rougeL 0.1867 0.1942  0.0075
  Qwen3.5    L3 bertscore_f1 0.2316 0.2741  0.0425
InternVL2    L2       rouge1 0.1313 0.2053  0.0739
InternVL2    L2       rouge2 0.0383 0.0564  0.0181
InternVL2    L2       rougeL 0.1270 0.1718  0.0447
InternVL2    L2 bertscore_f1 0.1133 0.1930  0.0797
InternVL2    L3       rouge1 0.0825 0.1613  0.0788
InternVL2    L3       rouge2 0.0058 0.0194  0.0136
InternVL2    L3       rougeL 0.0752 0.1359  0.0608
InternVL2    L3 bertscore_f1 0.0649 0.1906  0.1257


## Store to Supabase

In [26]:
df_eval.to_parquet("df_eval.parquet", index=True)
print(f"Saved df_eval.parquet: {df_eval.shape}")

from google.colab import files
files.download("df_eval.parquet")

Saved df_eval.parquet: (6750, 15)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [29]:
STORE_COLS = [
    "model_name", "dashboard_name", "metadata_id",
    "chart_id", "chart_title", "level",
    "generated", "reference",
    "generation_success", "degenerate_pattern",
    "rouge1", "rouge2", "rougeL", "bertscore_f1",
]

# Deduplicate degen_lookup on key columns to prevent fan-out on merge
degen_lookup = (
    df_vlm_raw_eval[["metadata_id","model_name","degenerate_pattern"]]
    .sort_values("degenerate_pattern", na_position="last")
    .drop_duplicates(subset=["metadata_id","model_name"], keep="first")
)
df_final = df_eval.merge(degen_lookup, on=["metadata_id","model_name"], how="left")

# Rename rougeL → rougel to match PostgreSQL column (case-insensitive)
df_final = df_final.rename(columns={"rougeL": "rougel"})
STORE_COLS = [c if c != "rougeL" else "rougel" for c in STORE_COLS]

df_scores = df_final[[c for c in STORE_COLS if c in df_final.columns]].copy()
df_scores["metadata_id"] = df_scores["metadata_id"].astype(str)

records = (
    df_scores
    .replace({float("nan"): None, float("inf"): None, float("-inf"): None})
    .to_dict(orient="records")
)
response = supabase.table("eval_scores").upsert(
    records,
    on_conflict="model_name,metadata_id,chart_id,level"
).execute()

print(f"Done. Upserted {len(records)} rows.")

Done. Upserted 6750 rows.
